In [1]:
import pandas as pd
from sqlalchemy import create_engine
import json

# 

## Task 1 - Nombre d'appels par trimestre

In [3]:
df_client = pd.read_sql("SELECT * FROM test_sql.test_client LIMIT 3", engine)
df_call = pd.read_sql("SELECT * FROM test_sql.test_call LIMIT 3", engine)

print(df_client.head)
print(df_call.head)

<bound method NDFrame.head of    id firstname lastname phonenumber creationdate
0   1    Younes      Ziz  0620324525   2023-01-24
1   2    Océana      Ben  0626272084   2023-03-01
2   3  Frederic      Mic  0639267901   2023-03-01>
<bound method NDFrame.head of       id  clientphonenumberin clientphonenumberout    calldate  \
0  13483            637083600                 None  2023-04-20   
1  30443            642699972                 None  2023-05-25   
2   4722            636420482                 None  2023-03-22   

   calldurationinseconds  
0                  520.0  
1                    6.0  
2                  161.0  >


In [4]:
query_first_task = """
SELECT
    CASE 
        WHEN calldate BETWEEN '2023-01-01' AND '2023-03-30' THEN 'Q1_2023'
        WHEN calldate BETWEEN '2023-04-01' AND '2023-06-30' THEN 'Q2_2023'
    END as quarter,
    COUNT(*) as nb_calls,
    SUM(CASE WHEN clientphonenumberin IS NOT NULL OR (clientphonenumberin IS NULL AND clientphonenumberout IS NULL) THEN 1 ELSE 0 END) as nb_calls_inbound,
    SUM(CASE WHEN clientphonenumberout IS NOT NULL THEN 1 ELSE 0 END) as nb_calls_outbound
FROM test_sql.test_call
WHERE calldate BETWEEN '2023-01-01' AND '2023-03-30'
   OR calldate BETWEEN '2023-04-01' AND '2023-06-30'
GROUP BY quarter
ORDER BY quarter
"""

In [5]:
df_first_task = pd.read_sql(query_first_task, engine)

result = {
    row['quarter']: {
        "nb_calls": int(row['nb_calls']),
        "nb_calls_inbound": int(row['nb_calls_inbound']),
        "nb_calls_outbound": int(row['nb_calls_outbound'])
    }
    for _, row in df_first_task.iterrows()
}

with open("data/calls_by_quarter.json", "w") as f:
    json.dump(result, f, indent=4)

print(json.dumps(result, indent=4))

{
    "Q1_2023": {
        "nb_calls": 7013,
        "nb_calls_inbound": 6989,
        "nb_calls_outbound": 24
    },
    "Q2_2023": {
        "nb_calls": 32646,
        "nb_calls_inbound": 29663,
        "nb_calls_outbound": 2983
    }
}


## Task 2 - Statistiques d'appels par client

In [6]:
# query = """
# SELECT 
#     column_name, 
#     data_type 
# FROM information_schema.columns 
# WHERE table_schema = 'test_sql' 
# AND table_name IN ('test_client', 'test_call')
# ORDER BY table_name, column_name
# """

# df_types = pd.read_sql(query, engine)
# print(df_types)

In [7]:
# # Exemples de phonenumber dans test_client
# print(pd.read_sql("SELECT phonenumber FROM test_sql.test_client LIMIT 5", engine))

# # Exemples de clientphonenumberin dans test_call
# print(pd.read_sql("SELECT clientphonenumberin, clientphonenumberout FROM test_sql.test_call LIMIT 5", engine))

In [8]:
query_second_task = """
SELECT
    CONCAT(c.firstname, ' ', c.lastname) as client_full_name,
    COUNT(ca.id) as nb_calls,
    SUM(ca.calldurationinseconds) as total_seconds,
    MAX(ca.calldate) as most_recent_call
FROM test_sql.test_client c
JOIN test_sql.test_call ca 
    ON ca.clientphonenumberin::text = SUBSTRING(c.phonenumber, 2)
    OR ca.clientphonenumberout::text = SUBSTRING(c.phonenumber, 2)
GROUP BY c.firstname, c.lastname
ORDER BY total_seconds DESC
"""

In [9]:
def seconds_to_hhmmss(seconds):
    seconds = int(seconds)
    h = seconds // 3600
    m = (seconds % 3600) // 60
    s = seconds % 60
    return f"{h:02d}:{m:02d}:{s:02d}"

In [10]:
df_second_task = pd.read_sql(query_second_task, engine)

# Convertir les secondes en HH:MM:SS
df_second_task['total_duration_calls'] = df_second_task['total_seconds'].apply(seconds_to_hhmmss)

df = df_second_task[['client_full_name', 'nb_calls', 'total_duration_calls', 'most_recent_call']]

df.to_csv("data/clients_calls.csv", index=False)
print(df.head())

  client_full_name  nb_calls total_duration_calls most_recent_call
0   Marie Jose Ast        19             05:50:56       2023-06-21
1      Mankoto San        47             04:34:30       2023-06-13
2      Valerie Lin        30             04:23:36       2023-03-24
3      Florian Nav         3             04:19:15       2023-06-19
4       angela Bel        15             04:14:08       2023-06-07
